## Scraping AWOIAF Characters
End-to-end scrape of character data from [A Wiki of Ice and Fire](https://awoiaf.westeros.org):

1. **List**: pull every character name from `List_of_characters` → `characters.csv`
2. **Enrich**: visit each character's page, parse the infobox (`father`, `mother`, `spouse`, `lover`, `issue`, `family`, `allegiance`) and collect every internal `/index.php/` link from the article body into `affiliated` (used as edges for the network analysis) → `characters_enriched.csv`

In [15]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from urllib.parse import quote
from concurrent.futures import ThreadPoolExecutor

### Setup
Full browser User-Agent — the wiki returns 403 for generic bot strings.

`Family` is included in the output schema for completeness; the AWOIAF infobox doesn't actually use that label (houses go in `Allegiance`), so the column will be empty for every row.

In [16]:
BASE = 'https://awoiaf.westeros.org'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

INFOBOX_LABELS = {
    'Father': 'father',
    'Mother': 'mother',
    'Spouse': 'spouse',
    'Lover': 'lover',
    'Lovers': 'lover',
    'Paramour': 'lover',
    'Issue': 'issue',
    'Family': 'family',
    'Allegiance': 'allegiance',
}

session = requests.Session()
session.headers.update(headers)

## 1. Scrape the character list
Character names appear as `<li>` items inside `mw-parser-output`, each wrapped in an `<a>` to a `/index.php/...` page. We skip the table-of-contents anchor links (`#A`, `#B`, ...).

In [17]:
list_page = session.get(BASE + '/index.php/List_of_characters', timeout=20)
print(f'Status code: {list_page.status_code}')

soup = BeautifulSoup(list_page.content, 'html.parser')
content = soup.find('div', class_='mw-parser-output')

character_names = []
for li in content.find_all('li'):
    a = li.find('a')
    if a and a.get('href', '').startswith('/index.php/'):
        name = a.get_text(strip=True)
        if name and name not in character_names:
            character_names.append(name)

pd.DataFrame(character_names, columns=['name']).to_csv('characters.csv', index=False)
print(f'Saved {len(character_names)} characters to characters.csv')
print(character_names[:10])

Status code: 200
Saved 3562 characters to characters.csv
['A certain man', 'Abelar Hightower', 'Abelon', 'Addam of Duskendale', 'Addam Frey', 'Addam Hightower', 'Addam Marbrand', 'Addam Osgrey', 'Addam Rivers', 'Addam Velaryon']


## 2. Enrich each character
### Helpers

In [18]:
PREFIX = '/index.php/'


def name_to_slug(name):
    """AWOIAF page slug: spaces -> underscores, then percent-encode the rest.

    Returned as a bare slug (no /index.php/ prefix). Prepend PREFIX when you need the URL path.
    """
    return quote(name.replace(' ', '_'), safe="_/(),")


def href_to_slug(href):
    """Return the slug for an internal wiki link, or None if the href isn't one we want.

    Skips red links (missing pages), edit/action links, and Special: pages.
    """
    if not href.startswith(PREFIX):
        return None
    if 'redlink=1' in href or 'action=' in href:
        return None
    slug = href[len(PREFIX):]
    if slug.startswith('Special:'):
        return None
    return slug


def cell_links_or_text(td):
    """Semicolon-joined slugs in the cell. Falls back to plain text (e.g. 'Unknown') if none."""
    slugs = []
    seen = set()
    for a in td.find_all('a'):
        slug = href_to_slug(a.get('href', ''))
        if slug is None or slug in seen:
            continue
        seen.add(slug)
        slugs.append(slug)
    if slugs:
        return ';'.join(slugs)
    text = td.get_text(' ', strip=True)
    return text if text else ''


def parse_infobox(soup):
    data = {col: '' for col in INFOBOX_LABELS.values()}
    infobox = soup.find('table', class_='infobox')
    if infobox is None:
        return data
    for row in infobox.find_all('tr'):
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        label = th.get_text(strip=True).rstrip(':')
        col = INFOBOX_LABELS.get(label)
        if col is None:
            continue
        data[col] = cell_links_or_text(td)
    return data


def parse_affiliated(soup, self_slug):
    """All unique internal-link slugs in the article body, in document order, excluding the page itself."""
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return ''
    slugs = []
    seen = set()
    for a in content.find_all('a'):
        slug = href_to_slug(a.get('href', ''))
        if slug is None or slug == self_slug or slug in seen:
            continue
        seen.add(slug)
        slugs.append(slug)
    return ';'.join(slugs)

### Per-character scrape

In [19]:
EMPTY = {'father': '', 'mother': '', 'spouse': '', 'lover': '', 'issue': '',
         'family': '', 'allegiance': '', 'affiliated': ''}


def scrape_character(name):
    slug = name_to_slug(name)
    try:
        resp = session.get(BASE + PREFIX + slug, timeout=20)
    except requests.RequestException as e:
        print(f'  ! request failed for {name}: {e}')
        return {'name': name, 'ID': slug, **EMPTY}
    if resp.status_code != 200:
        print(f'  ! {resp.status_code} for {name}')
        return {'name': name, 'ID': slug, **EMPTY}
    soup = BeautifulSoup(resp.content, 'html.parser')
    info = parse_infobox(soup)
    return {
        'name': name,
        'ID': slug,
        'father': info['father'],
        'mother': info['mother'],
        'spouse': info['spouse'],
        'lover': info['lover'],
        'issue': info['issue'],
        'family': info['family'],
        'allegiance': info['allegiance'],
        'affiliated': parse_affiliated(soup, slug),
    }

### Run the scrape
~3,500 characters, parallelized across 8 threads via `ThreadPoolExecutor`. `executor.map` preserves input order, so the output CSV stays alphabetical and checkpoints every 100 rows remain coherent. Tune `WORKERS` if the wiki starts throttling.

In [20]:
characters = pd.read_csv('characters.csv')['name'].tolist()
print(f'{len(characters)} characters to enrich')

COLUMNS = ['name', 'ID', 'father', 'mother', 'spouse', 'lover', 'issue', 'family', 'allegiance', 'affiliated']
OUT = 'characters_enriched.csv'
WORKERS = 8

rows = []
with ThreadPoolExecutor(max_workers=WORKERS) as executor: # Speeds up runtime by 8x buy parallelizing requests
    for i, row in enumerate(executor.map(scrape_character, characters), 1):
        rows.append(row)
        if i % 50 == 0:
            print(f'  {i}/{len(characters)}')
        if i % 100 == 0:
            pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)

pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)
print(f'Saved {len(rows)} rows to {OUT}')

3562 characters to enrich
  50/3562
  100/3562
  150/3562
  200/3562
  250/3562
  300/3562
  350/3562
  400/3562
  450/3562
  500/3562
  550/3562
  600/3562
  650/3562
  700/3562
  750/3562
  800/3562
  850/3562
  900/3562
  950/3562
  1000/3562
  1050/3562
  1100/3562
  1150/3562
  1200/3562
  1250/3562
  ! 404 for Gwayne Gardener
  1300/3562
  1350/3562
  1400/3562
  1450/3562
  1500/3562
  1550/3562
  1600/3562
  1650/3562
  1700/3562
  1750/3562
  1800/3562
  1850/3562
  1900/3562
  1950/3562
  ! 404 for Lyman
  2000/3562
  2050/3562
  2100/3562
  2150/3562
  2200/3562
  2250/3562
  2300/3562
  2350/3562
  2400/3562
  2450/3562
  ! 404 for Pate the stonemason
  2500/3562
  2550/3562
  2600/3562
  2650/3562
  2700/3562
  2750/3562
  2800/3562
  2850/3562
  2900/3562
  2950/3562
  3000/3562
  3050/3562
  3100/3562
  3150/3562
  3200/3562
  3250/3562
  3300/3562
  3350/3562
  3400/3562
  3450/3562
  3500/3562
  3550/3562
Saved 3562 rows to characters_enriched.csv


### Quick sanity check

In [23]:
df = pd.read_csv(OUT)
print(df.shape)
df[df['name'].isin(['Eddard Stark', 'Jon Snow', 'Tyrion Lannister'])]
df[0:20]

(3562, 10)


,name,ID,father,mother,spouse,lover,issue,family,allegiance,affiliated
0,A certain man,A_certain_man,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Myr;A_Clash_of_Kings;Game_of_Thrones;Game_of_T...
1,Abelar Hightower,Abelar_Hightower,NaN,NaN,NaN,NaN,NaN,NaN,House_Hightower,House_Hightower;File:The_Dragon_and_the_Tower_...
2,Abelon,Abelon,NaN,NaN,NaN,NaN,NaN,NaN,Citadel,Maesters;Archmaester;Citadel;Westeros;Fire_%26...
3,Addam of Duskendale,Addam_of_Duskendale,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Addam;Crownlands;The_World_of_Ice_%26_Fire;Dus...
4,Addam Frey,Addam_Frey,NaN,NaN,NaN,NaN,NaN,NaN,House_Frey,Addam;House_Frey;Knight;Rivermen;Twins;The_Mys...
5,Addam Hightower,Addam_Hightower,Manfred_Hightower_(Aegon%27s_Conquest),NaN,Unknown,NaN,Manfred_Hightower_(son_of_Addam);Patrice_Hight...,NaN,NaN,Addam;House_Hightower;Knight;Beacon_of_the_Sou...
6,Addam Marbrand,Addam_Marbrand,Damon_Marbrand,NaN,NaN,NaN,NaN,NaN,NaN,Addam;House_Marbrand;City_Watch_of_King%27s_La...
7,Addam Osgrey,Addam_Osgrey,Eustace_Osgrey,NaN,NaN,NaN,NaN,NaN,House_Osgrey,Addam;House_Osgrey;Reach;Years_after_Aegon%27s...
8,Addam Rivers,Addam_Rivers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Addam;Knight;King_of_the_Trident;Rivermen;The_...
9,Addam Velaryon,Addam_Velaryon,NaN,Marilda_of_Hull,NaN,NaN,NaN,NaN,NaN,Addam;House_Velaryon;File:Addam_Velaryon_by_Ch...
